# **makemore MLP**

Implementing the **makemore** bigram model through a **Multi-Layer Perceptron (MLP)** as discussed in *Bengio et al. (2023)*

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

Reading in all the words (names):

In [3]:
words = open('names.txt', 'r').read().splitlines()
words[:8], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

Build the **vocabulary of characters** and mapps to/from integers:

In [9]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s  in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


Building the **dataset** we can then train the model on, this time we use a **context** window: meaning the number of tokens (here single characters) we use to predict the next character:
- `block_size` = context window of characters used to predict the next character

In [12]:
block_size = 3
X, Y = [], []

for w in words[:3]:
    print(w)

    # Initialize an empty context
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix) # We are storing character IDs
        print(''.join(itos[i] for i in context), '-->', itos[ix])

        # Update context: crop current and append new index
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)        

emma
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
olivia
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
ava
... --> a
..a --> v
.av --> a
ava --> .


Let us now look at our matrices of predictors `X` and labels `Y`:
- `X.shape` is (16, 3) since we have a 3-long context window, and we happen to have 16 training "examples" for those 3 words
- `Y.shape` is just 16 one-character ID labels for the 16 training examples we happen to have for the first three words

In [15]:
X, Y, X.shape, Y.shape

(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1]]),
 tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0]),
 torch.Size([16, 3]),
 torch.Size([16]))

#### **Creating the Embedding Matrix for our Characters**

The embedding matrix will be comprised of 27 rows (for 27 characters) and `d` dimensions as columns (here we start simple with `d` = 2):
- `C = torch.randn((27, 2))` holds the parameters, it has one learnable vector per vocabulary entry, and it's the only place the embeddings actually live. Training updates `C`
- `emb` is just the result of a lookup for our particular training data. If the same character shows up many times in `X`, all those positions in `emb` are copies of the same row of `C`

In [18]:
C = torch.randn((27, 2))
C

tensor([[ 0.1292, -0.0182],
        [-0.1804, -0.7091],
        [ 0.3504,  0.4403],
        [ 1.3698, -0.3583],
        [ 1.3619,  0.9207],
        [ 1.0490, -0.2331],
        [-0.1680,  0.3808],
        [ 0.4951, -0.8115],
        [ 0.6038, -1.5593],
        [-0.6077,  1.2182],
        [ 0.3492,  0.8138],
        [-0.1616,  0.0561],
        [-0.0920,  0.7264],
        [-0.2911,  1.0965],
        [ 0.5518, -0.9816],
        [ 0.0319, -0.3395],
        [-0.2886,  0.1666],
        [ 1.1510,  1.6152],
        [ 0.5516, -1.1047],
        [ 0.8372,  0.1692],
        [-0.0443, -0.8831],
        [-0.2041,  0.7493],
        [-0.2427,  1.0686],
        [ 2.5083, -0.8530],
        [ 0.8448, -0.4766],
        [ 0.0527,  0.4206],
        [-0.1314,  0.2318]])

In [20]:
C[X].shape

torch.Size([16, 3, 2])

In [22]:
emb = C[X]
emb

tensor([[[ 0.1292, -0.0182],
         [ 0.1292, -0.0182],
         [ 0.1292, -0.0182]],

        [[ 0.1292, -0.0182],
         [ 0.1292, -0.0182],
         [ 1.0490, -0.2331]],

        [[ 0.1292, -0.0182],
         [ 1.0490, -0.2331],
         [-0.2911,  1.0965]],

        [[ 1.0490, -0.2331],
         [-0.2911,  1.0965],
         [-0.2911,  1.0965]],

        [[-0.2911,  1.0965],
         [-0.2911,  1.0965],
         [-0.1804, -0.7091]],

        [[ 0.1292, -0.0182],
         [ 0.1292, -0.0182],
         [ 0.1292, -0.0182]],

        [[ 0.1292, -0.0182],
         [ 0.1292, -0.0182],
         [ 0.0319, -0.3395]],

        [[ 0.1292, -0.0182],
         [ 0.0319, -0.3395],
         [-0.0920,  0.7264]],

        [[ 0.0319, -0.3395],
         [-0.0920,  0.7264],
         [-0.6077,  1.2182]],

        [[-0.0920,  0.7264],
         [-0.6077,  1.2182],
         [-0.2427,  1.0686]],

        [[-0.6077,  1.2182],
         [-0.2427,  1.0686],
         [-0.6077,  1.2182]],

        [[-0.2427,  1

**Reminder:** We are aiming to input 3 characters and predict the next character:
- Our input size is therefore 3 embeddings x with 2 dimensions each, thus `3x2` = `6`
- We arbitrarily choose the `hidden_units` (number of neurons in the hidden layer) as 100
- The number of weights we need is thus `(6, 100)`, 6 inputs x 100 hidden neurons = 600 different weights
- Remember there is a single bias per hidden neuron, so we only need to initialize a 100 of them

In [24]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [25]:
emb.shape, W1.shape

(torch.Size([16, 3, 2]), torch.Size([6, 100]))

At this stage, we would normally want to matrix multiply our embeddings (inputs) with the weights: `emb @ W1 + b1`, however there is a clear **shape mismatch** currently:
- We want to multiple **emb** `(16, 3, 2)` tensor with **W1** `(6, 100)`
- To solve this efficiently, we use `torch.view(16, 6)` or equivalently `torch.reshape(16, 6)` on `emb`

In [28]:
emb = emb.view(emb.shape[0], 6)
emb.shape

torch.Size([16, 6])

`h` is the hidden layer activations, therefore we also wrap the multiplication with the **`tanh`** non-linear activation function:

In [32]:
h = torch.tanh(emb @ W1 + b1)
h, h.shape

(tensor([[ 0.8743,  0.0388, -0.9482,  ..., -0.7742, -0.1538,  0.7225],
         [ 0.6767,  0.0176, -0.9858,  ..., -0.6237, -0.9206,  0.9709],
         [ 0.9260, -0.3406, -0.9528,  ..., -0.8527,  0.7873, -0.9591],
         ...,
         [ 0.9375,  0.5196, -0.9196,  ..., -0.9611, -0.2512,  0.9495],
         [ 0.5919, -0.9584, -0.6865,  ...,  0.1689,  0.6744, -0.7852],
         [ 0.9673,  0.9948, -0.9247,  ..., -0.9750,  0.0429,  0.9764]]),
 torch.Size([16, 100]))

#### **Output Layer**

The output layer is expected to have 27 logits (for the 27 characters in our vocabulary) per example (remember we have 16 examples here, so `logits.shape` = `16, ), which are then `SoftMax`-ed to obtain probabilities.

In [33]:
W2 = torch.randn(100, 27)
b2 = torch.randn(27)

In [34]:
h.shape, W2.shape

(torch.Size([16, 100]), torch.Size([100, 27]))

We can now obtain the logits: 27 logits for each of the 16 examples, therefore `logits.shape` = `(16, 27)`:

In [35]:
logits = h @ W2 + b2
logits.shape

torch.Size([16, 27])

**Transforming the logits to probabilities:**

Evidently, for now, these are **untrained** weights, and therefore random probabilities.

In [37]:
counts = logits.exp()

probs = counts / counts.sum(1, keepdim=True)
probs.shape, probs

(torch.Size([16, 27]),
 tensor([[1.0511e-07, 5.7792e-08, 1.2033e-07, 1.4956e-02, 9.0602e-08, 2.4366e-04,
          1.0639e-08, 7.4085e-07, 2.7078e-02, 3.2441e-05, 1.2316e-09, 1.2773e-06,
          5.3785e-07, 1.7887e-03, 8.3341e-09, 9.4172e-01, 4.1302e-09, 2.1422e-05,
          1.5045e-03, 4.6991e-10, 3.5277e-07, 1.2567e-02, 4.1589e-07, 7.6761e-09,
          1.4754e-06, 8.2518e-07, 8.1596e-05],
         [9.4659e-10, 3.7572e-06, 8.9978e-08, 8.3163e-05, 4.5896e-06, 6.3965e-05,
          1.1051e-08, 5.9912e-05, 1.6080e-01, 2.5021e-07, 1.8843e-09, 1.7840e-03,
          1.2139e-08, 1.3656e-09, 3.9997e-10, 7.8988e-01, 5.0546e-08, 1.0299e-02,
          4.0593e-06, 1.4570e-12, 7.7798e-07, 3.6904e-02, 1.3159e-11, 2.8632e-08,
          8.3426e-05, 3.2350e-05, 2.8027e-06],
         [4.5920e-05, 3.3291e-10, 8.4573e-05, 3.5738e-02, 1.9902e-11, 3.2009e-01,
          1.5250e-08, 4.0235e-09, 1.3680e-01, 9.1563e-06, 4.0339e-06, 3.6145e-04,
          3.1934e-02, 2.9799e-04, 1.2115e-03, 3.5083e-03, 1.230

In [38]:
probs.shape

torch.Size([16, 27])

We now want to look at the rows of `probs` and extract the predicted probability assigned to the true next character:
- For instance, we want to see what the model predicted as a probability to get `m` as the next character given the input `..e`
- To achieve this, we must acces, the second row of `probs`, and the idx of `m`

In [51]:
probs[1, Y[13]]

tensor(1.3159e-11)

More generally, we can just iterate through all of our examples:

In [52]:
probs[torch.arange(16), Y]

tensor([2.4366e-04, 1.3656e-09, 2.9799e-04, 2.3562e-14, 1.3281e-11, 9.4172e-01,
        9.7515e-07, 1.2485e-04, 1.1526e-07, 3.3165e-12, 5.7148e-16, 2.3123e-11,
        5.7792e-08, 1.6256e-03, 1.2772e-10, 5.2056e-10])

#### **Loss Function: Negative Log-Likelihood**:

This is the loss we want to **minimize** when training the MLP neural network:

In [54]:
loss = -probs[torch.arange(16), Y].log().mean()
loss

tensor(17.8378)